In [1]:
!pip install -U bitsandbytes transformers accelerate qwen-vl-utils zeus-ml -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 15.7 MB/s eta 0:00:00


In [2]:
from kaggle_secrets import UserSecretsClient
import os

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF token loaded:", os.environ["HF_TOKEN"][:8], "...")

HF token loaded: hf_GibuC ...


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Visible devices:", torch.cuda.device_count())
print("Device:", torch.cuda.get_device_name(0))

CUDA available: True
Visible devices: 1
Device: Tesla T4


In [5]:
from zeus.monitor import ZeusMonitor
monitor = ZeusMonitor(gpu_indices=[0])
print("Idle power (W):", monitor.gpus.getInstantPowerUsage(0) / 1000)

/opt/rocm/lib/libamd_smi.so: cannot open shared object file: No such file or directory
Unable to find libamd_smi.so library try installing amd-smi-lib from your package manager
[2026-07-22 22:26:26,472] [zeus.device.gpu.nvidia](nvidia.py:47) pynvml is available and initialized.
[2026-07-22 22:26:26,473] [zeus.device.cpu.rapl](rapl.py:137) RAPL is not supported on this CPU.
[2026-07-22 22:26:26,474] [zeus.monitor.energy](energy.py:209) Monitoring GPU indices [0].
[2026-07-22 22:26:26,475] [zeus.monitor.energy](energy.py:210) Monitoring CPU indices []
Idle power (W): 9.868


In [6]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Loaded on:", next(model.parameters()).device)
print("GPU memory (GB):", torch.cuda.memory_allocated(0) / 1e9)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

... (2 outputs left out)

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded on: cuda:0
GPU memory (GB): 5.911530496


In [7]:
# Cell 6 — define constants and paths:

import os, json, csv, time
from datetime import datetime, timezone

MODEL_NAME = "qwen2.5-vl-7b"
DATA_DIR = "/kaggle/input/notebooks/minhajurrahman223050/ecovlm-data-prep/vqav2_subset"
OUT_DIR = "/kaggle/working"
LOG_FILE = os.path.join(OUT_DIR, f"{MODEL_NAME}_telemetry.csv")
MANIFEST_FILE = os.path.join(OUT_DIR, f"{MODEL_NAME}_manifest.json")

N_RUNS = 5
MAX_NEW_TOKENS = 32
VQA_PROMPT_SUFFIX = "Answer the question using a single word or phrase."

with open(os.path.join(DATA_DIR, "metadata.json")) as f:
    SAMPLES = json.load(f)

print(f"Samples loaded: {len(SAMPLES)}")
print(f"Log file: {LOG_FILE}")

Samples loaded: 1000
Log file: /kaggle/working/qwen2.5-vl-7b_telemetry.csv


In [8]:
# Cell 7 — define VQA accuracy scorer:

def vqa_accuracy(predicted: str, ground_truth_answers: list) -> float:
    """
    Official VQA v2 accuracy:
    min(#humans who gave this answer / 3, 1.0)
    Normalise to lowercase, strip punctuation and articles.
    """
    import re
    def normalise(s):
        s = s.lower().strip()
        s = re.sub(r"[^\w\s]", "", s)
        for article in ["a ", "an ", "the "]:
            if s.startswith(article):
                s = s[len(article):]
        return s.strip()

    pred_norm = normalise(predicted)
    matches = sum(1 for a in ground_truth_answers if normalise(a) == pred_norm)
    return min(matches / 3, 1.0)

In [9]:
# Cell 8 — the manifest (resume mechanism):
def load_manifest():
    if os.path.exists(MANIFEST_FILE):
        with open(MANIFEST_FILE) as f:
            return json.load(f)
    return {"completed_runs": [], "current_run": None}

def save_manifest(manifest):
    with open(MANIFEST_FILE, "w") as f:
        json.dump(manifest, f, indent=2)

manifest = load_manifest()
print("Completed runs so far:", manifest["completed_runs"])

Completed runs so far: []


In [10]:
# Cell 9 — measure idle baseline:
import time

print("Measuring idle baseline (30 seconds)...")
idle_readings = []
for _ in range(300):   # 300 × 100ms = 30 seconds
    idle_readings.append(monitor.gpus.getInstantPowerUsage(0) / 1000)
    time.sleep(0.1)

IDLE_POWER_W = sum(idle_readings) / len(idle_readings)
print(f"Idle baseline: {IDLE_POWER_W:.3f} W")
print(f"Idle std dev:  {(sum((x-IDLE_POWER_W)**2 for x in idle_readings)/len(idle_readings))**0.5:.3f} W")

Measuring idle baseline (30 seconds)...
Idle baseline: 27.985 W
Idle std dev:  0.120 W


In [11]:
# Cell 10 — the measurement loop
from PIL import Image
import csv as _csv

# initialise CSV if it doesn't exist yet
write_header = not os.path.exists(LOG_FILE)
log_f = open(LOG_FILE, "a", newline="", buffering=1)  # line-buffered = crash-safe
writer = csv.DictWriter(log_f, fieldnames=[
    "run_id", "sample_idx", "question_id", "image_id",
    "question", "predicted", "vqa_accuracy",
    "energy_gross_j", "energy_net_j", "avg_power_w",
    "idle_power_w", "elapsed_s", "tokens_generated",
    "timestamp"
])
if write_header:
    writer.writeheader()

IMG_DIR = os.path.join(DATA_DIR, "images")

for run_id in range(1, N_RUNS + 1):
    if run_id in manifest["completed_runs"]:
        print(f"Run {run_id} already complete — skipping.")
        continue

    print(f"\n{'='*50}")
    print(f"Starting run {run_id} of {N_RUNS}")
    print(f"{'='*50}")
    manifest["current_run"] = run_id
    save_manifest(manifest)

    # resume mid-run: skip samples already logged for this run
    done_idx = set()
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE) as _f:
            for row in _csv.DictReader(_f):
                if int(row["run_id"]) == run_id:
                    done_idx.add(int(row["sample_idx"]))
    if done_idx:
        print(f"Resuming run {run_id}: {len(done_idx)} samples already logged, skipping those.")

    for s in SAMPLES:
        idx = s["sample_idx"]
        if idx in done_idx:
            continue

        image = Image.open(os.path.join(IMG_DIR, s["image_file"])).convert("RGB")

        messages = [{
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": f"{s['question']} {VQA_PROMPT_SUFFIX}"},
            ],
        }]

        text_prompt = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = processor(
            text=[text_prompt], images=[image], return_tensors="pt"
        ).to("cuda:0")

        # --- energy window ---
        window_name = f"run{run_id}_sample{idx}"
        monitor.begin_window(window_name)
        t0 = time.perf_counter()

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
            )

        elapsed = time.perf_counter() - t0
        measurement = monitor.end_window(window_name)
        # ---------------------

        generated_ids = output_ids[:, inputs.input_ids.shape[1]:]
        predicted = processor.batch_decode(
            generated_ids, skip_special_tokens=True
        )[0].strip()
        tokens_generated = generated_ids.shape[1]

        energy_gross = measurement.total_energy   # joules, from Zeus
        avg_power    = energy_gross / elapsed if elapsed > 0 else 0
        energy_net   = max((avg_power - IDLE_POWER_W) * elapsed, 0)
        acc          = vqa_accuracy(predicted, s["answers"])

        writer.writerow({
            "run_id": run_id,
            "sample_idx": idx,
            "question_id": s["question_id"],
            "image_id": s["image_id"],
            "question": s["question"],
            "predicted": predicted,
            "vqa_accuracy": round(acc, 4),
            "energy_gross_j": round(energy_gross, 4),
            "energy_net_j": round(energy_net, 4),
            "avg_power_w": round(avg_power, 3),
            "idle_power_w": round(IDLE_POWER_W, 3),
            "elapsed_s": round(elapsed, 4),
            "tokens_generated": tokens_generated,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        })

    manifest["completed_runs"].append(run_id)
    manifest["current_run"] = None
    save_manifest(manifest)
    print(f"Run {run_id} complete.")

log_f.close()
print("\nAll runs complete.")
print(f"Total rows logged: {sum(1 for _ in open(LOG_FILE)) - 1}")


Starting run 1 of 5
[2026-07-22 22:29:10,112] [zeus.utils.framework](framework.py:25) PyTorch with CUDA support is available.
Run 1 complete.

Starting run 2 of 5
Run 2 complete.

Starting run 3 of 5
Run 3 complete.

Starting run 4 of 5
Run 4 complete.

Starting run 5 of 5
Run 5 complete.

All runs complete.
Total rows logged: 5000


In [12]:
import shutil
backup = LOG_FILE.replace(".csv", "_backup.csv")
shutil.copy(LOG_FILE, backup)
print("Rows in log:", sum(1 for _ in open(LOG_FILE)) - 1)
print("Backup written:", backup, os.path.exists(backup))

Rows in log: 5000
Backup written: /kaggle/working/qwen2.5-vl-7b_telemetry_backup.csv True
